<a href="https://colab.research.google.com/github/ugisrutinsRSU/RSU_Colab/blob/main/02_Intro_to_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to PyTorch

In this notebook you will learn the fundamentals of PyTorch — the most widely used deep learning framework in research and industry.

By the end of this notebook you will be able to:
- Create and manipulate **tensors** (PyTorch's core data structure)
- Reshape tensors confidently
- Build a **custom Dataset** and load data with a **DataLoader**
- Apply **data augmentation** and preprocessing transforms

---
📌 **Prerequisites:** basic Python and NumPy familiarity.

## 1. What is PyTorch?

[PyTorch](https://pytorch.org/) is an open-source deep learning framework developed by Meta AI. It is built around two key ideas:

1. **Tensor computation** — like NumPy, but with GPU acceleration.
2. **Automatic differentiation** — PyTorch tracks operations on tensors so it can compute gradients automatically (you will use this heavily when training models in the next notebook).

To install PyTorch, visit https://pytorch.org/get-started/locally/ and follow the instructions for your system.

```bash
# Typical CPU-only install
pip install torch torchvision
```

## 2. Working with Tensors

A **tensor** is the fundamental data structure in PyTorch — think of it as a generalisation of a __________ array:

| Concept | NumPy | PyTorch |
|---------|-------|---------|
| 1-D array | `np.array([1,2,3])` | `torch.tensor([1,2,3])` |
| 2-D array | `np.zeros((3,4))` | `torch.zeros((3,4))` |
| N-D array | `ndarray` | `Tensor` |

The key advantage over NumPy: tensors can live on a **GPU** and support automatic differentiation.

### 2.1 Creating Tensors

Fill in the missing function names to create each type of tensor:

In [1]:
import torch

# Izveido tenzoru ar 2x3 izmēru, kas aizpildīts ar nejaušām vērtībām no vienmērīgā sadalījuma [0, 1)
tensor_random = torch.rand((2, 3))

# Izveido tenzoru ar 4x4 izmēru, kas aizpildīts ar nullēm
tensor_zeros = torch.zeros((4, 4))

# Izveido tenzoru ar 3x3 izmēru, kas aizpildīts ar vieniniekiem
tensor_ones = torch.ones((3, 3))

# Izveido tenzoru tieši no Python saraksta
tensor_list = torch.tensor([[1, 2, 3], [4, 5, 6]])

print("Nejaušs tenzors:\n", tensor_random)
print("Nulļu tenzors:\n", tensor_zeros)
print("Vieninieku tenzors:\n", tensor_ones)
print("Tenzors no saraksta:\n", tensor_list)

Nejaušs tenzors:
 tensor([[0.7821, 0.1475, 0.5746],
        [0.4970, 0.5501, 0.4050]])
Nulļu tenzors:
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Vieninieku tenzors:
 tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
Tenzors no saraksta:
 tensor([[1, 2, 3],
        [4, 5, 6]])


### 2.2 Basic Tensor Operations

Fill in the correct operators and function name:

- Element-wise addition: `c = a ______ b`
- Element-wise multiplication: `d = a ______ b`
- Matrix multiplication: `e = torch.________(a, b)`

In [2]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

# Saskaitīšana pa elementiem — katrs elements tiek saskaitīts neatkarīgi
c = a + b

# Reizināšana pa elementiem (NAV matricas reizināšana)
d = a * b

# Matricas reizināšana — rindas × kolonnas, standarta lineārās algebras izpratnē
# Var izmantot arī "@" operatoru: e = a @ b
e = torch.matmul(a, b)

print("a + b:\n", c)
print("a * b (pa elementiem):\n", d)
print("a @ b (matmul):\n", e)

a + b:
 tensor([[ 6.,  8.],
        [10., 12.]])
a * b (pa elementiem):
 tensor([[ 5., 12.],
        [21., 32.]])
a @ b (matmul):
 tensor([[19., 22.],
        [43., 50.]])


### 2.3 Inspecting Tensor Properties

Three attributes you will check constantly when debugging:

- **`.shape`** — dimensions of the tensor (e.g. `torch.Size([2, 3])`)
- **`.dtype`** — element data type (e.g. `torch.float32`)
- **`.device`** — where the tensor lives (`cpu` or `cuda:0`)

Fill in the attribute names:

In [3]:
tensor = torch.rand(2, 3)

# .shape — tenzora dimensijas (piem., torch.Size([2, 3]))
print("Forma:", tensor.shape)
# .dtype — elementu datu tips (piem., torch.float32)
print("Datu tips:", tensor.dtype)
# .device — kur tenzors atrodas (cpu vai cuda:0)
print("Ierīce:", tensor.device)

Forma: torch.Size([2, 3])
Datu tips: torch.float32
Ierīce: cpu


### 2.4 Moving Tensors Between CPU and GPU

GPU training can be 10–100× faster than CPU for large models. PyTorch makes it easy to move data between devices. The standard pattern is to detect the available device once at the top of your script and then move everything there.

Fill in the missing method name and variable:

In [4]:
# Nosaka pieejamo ierīci — ja nav pieejams GPU, tiek izmantots CPU
# torch.cuda.is_available() pārbauda, vai ir pieejams CUDA saderīgs GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Izmantotā ierīce:", device)

# Pārvieto tenzoru uz izvēlēto ierīci (GPU, ja pieejams)
tensor_gpu = tensor.to(device)
print("Tenzora ierīce:", tensor_gpu.device)

Izmantotā ierīce: cpu
Tenzora ierīce: cpu


## 3. Tensor Reshaping

Reshaping is one of the most common operations you will perform — for example, flattening an image before passing it to a linear layer, or adding a batch dimension.

| Method | What it does |
|--------|--------------|
| `.view(shape)` | Reinterpret the data with a new shape — **zero-copy**, requires contiguous memory |
| `.reshape(shape)` | Same as view but works even if memory is non-contiguous (safer default) |
| `.squeeze()` | Remove all dimensions of size 1 |
| `.unsqueeze(dim)` | Insert a new dimension of size 1 at position `dim` |
| `.permute(dims)` | Reorder dimensions (useful when converting between channel-first and channel-last) |

💡 Use `-1` in a shape to let PyTorch infer that dimension automatically.

Fill in the blanks to perform the described reshape operations:

In [ ]:
t = torch.arange(24)   # 1-D tenzors: [0, 1, 2, ..., 23]
print("Sākotnējā forma:", t.shape)   # torch.Size([24])

# Pārveido par 2-D: 4 rindas, 6 kolonnas
t_2d = t.reshape(4, 6)
print("Pārveidots uz (4, 6):\n", t_2d)

# Saplacina atpakaļ uz 1-D — izmanto -1, lai PyTorch automātiski aprēķinātu izmēru
t_flat = t_2d.reshape(-1)
print("Saplacināts:", t_flat.shape)   # torch.Size([24])

# Pārveido par 3-D: partija=2, augstums=3, platums=4
t_3d = t.reshape(2, 3, 4)
print("Pārveidots uz (2, 3, 4):\n", t_3d)

Fill in the blanks for `squeeze` and `unsqueeze`:

In [ ]:
t = torch.rand(3, 1, 4)   # vidējā dimensija ir 1
print("Sākotnējā forma:", t.shape)          # (3, 1, 4)

# Noņem visas dimensijas, kuru izmērs ir 1
t_sq = t.squeeze()
print("Pēc squeeze:", t_sq.shape)        # (3, 4)

# Pievieno partijas (batch) dimensiju ar izmēru 1 pozīcijā 0
t_unsq = t_sq.unsqueeze(0)
print("Pēc unsqueeze(0):", t_unsq.shape) # (1, 3, 4)

Fill in the `permute` call to convert from channel-first `(B, C, H, W)` to channel-last `(B, H, W, C)`:

In [ ]:
# PyTorch izmanto "channel-first" formātu: (partija, kanāli, augstums, platums)
# Dažas bibliotēkas izmanto "channel-last": (partija, augstums, platums, kanāli)

img = torch.rand(8, 3, 32, 32)        # (B, C, H, W) -> (0, 1, 2, 3)
# Pārkārto dimensijas, lai kanāls (C) būtu pēdējais
img_hwc = img.permute(0, 2, 3, 1)  # → (B, H, W, C)
print("Channel-first:", img.shape)
print("Channel-last: ", img_hwc.shape)

## 4. Datasets and DataLoaders

PyTorch separates **what your data is** from **how it is served to the model**:

- A **`Dataset`** defines *what* your data looks like — it knows how many samples there are and how to retrieve the i-th one.
- A **`DataLoader`** wraps a Dataset and handles *how* data is delivered — batching, shuffling, parallel loading.

```
Dataset  →  DataLoader  →  Training loop
 (storage)    (batching)
```

A Dataset represents the __________ data, defining how each sample is stored and accessed.

A DataLoader helps in __________ the data into manageable batches for training.

### Why not just load everything into a list?
Real datasets (ImageNet: 1.2 M images) do not fit in RAM. The Dataset + DataLoader pattern lets you load only what you need, when you need it.

## 5. Creating a Custom Dataset

To create a custom Dataset, subclass `torch.utils.data.Dataset` and implement exactly **two methods**:

- `__len__()` — returns the total number of samples
- `__getitem__(index)` — returns the sample at position `index`

The DataLoader will call these methods internally. Fill in the missing return value:

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

class CustomImageDataset(Dataset):
    """
    Pielāgota datu kopa attēlu ielādei.
    Pārmanto no torch.utils.data.Dataset.
    """
    def __init__(self, image_dir, transform=None):
        """
        Args:
            image_dir (string): Ceļš uz direktoriju ar attēliem.
            transform (callable, optional): Pēc izvēles transformācija, ko piemērot paraugam.
        """
        self.image_dir = image_dir
        # Iegūst sarakstu ar visiem failu nosaukumiem norādītajā direktorijā
        self.image_files = os.listdir(image_dir)
        self.transform = transform

    def __len__(self):
        # DataLoader izmanto šo metodi, lai uzzinātu kopējo paraugu skaitu
        return len(self.image_files)

    def __getitem__(self, index):
        # Izveido pilnu ceļu uz attēla failu
        img_path = os.path.join(self.image_dir, self.image_files[index])
        # Atver attēlu un konvertē to uz RGB krāsu telpu
        image = Image.open(img_path).convert("RGB")

        # Ja ir definēta transformācija, piemēro to attēlam
        if self.transform:
            image = self.transform(image)

        return image

## 6. Using DataLoader

Key `DataLoader` parameters:

| Parameter | Description |
|-----------|-------------|
| `batch_size` | How many samples per batch |
| `shuffle` | Randomise order each epoch (use `True` for training, `False` for validation) |
| `num_workers` | Number of parallel processes for loading (0 = main process only) |

Fill in sensible values for a training DataLoader:

In [ ]:
# Lai šis kods darbotos, aizstājiet "path/to/images" ar reālu ceļu uz attēlu direktoriju.
# Piemēram, ja izveidojat mapi "my_images" un ievietojat tajā dažus attēlus:
# dataset = CustomImageDataset(image_dir="my_images")

# Izveido DataLoader, kas pārvaldīs datu ielādi
# batch_size=32: katrā partijā būs 32 attēli
# shuffle=True: dati tiks sajaukti katrā epohā (svarīgi trenēšanai)
# num_workers=2: datu ielāde notiks divos paralēlos procesos, lai paātrinātu
# dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

# # Iterē cauri datu partijām
# for batch in dataloader:
#     # Pēc transformācijām sagaidāmā formas dimensija: (partijas_izmērs, kanāli, augstums, platums)
#     print(batch.shape)  
#     break

print("Atkomentējiet kodu augstāk, kad jums būs reāla attēlu direktorija.")

## 7. Loading a Built-in Dataset (MNIST)

`torchvision.datasets` ships with many standard datasets (MNIST, CIFAR-10, ImageNet, …). This is the quickest way to get started without managing files yourself.

Before loading, we define a **transform pipeline** — a sequence of preprocessing steps applied to every sample on the fly.

Fill in the correct transform names and the expected dataset length:

In [ ]:
import torchvision.transforms as transforms
from torchvision import datasets

# Izveido transformāciju konveijeru, kas tiks piemērots katram attēlam
transform = transforms.Compose([
    # Pārvērš PIL attēlu (vērtības [0,255]) par float tenzoru (vērtības [0,1])
    transforms.ToTensor(),         
    # Normalizē tenzoru, lai vērtības būtu diapazonā [-1,1]
    # formula: (ievade - vidējā vērtība) / standartnovirze
    # (ievade - 0.5) / 0.5 = 2 * ievade - 1
    transforms.Normalize((0.5,), (0.5,)) 
])

# Ielādē MNIST treniņkopu. Ja tā nav lokāli pieejama, tā tiks lejupielādēta.
train_dataset = datasets.MNIST(root='data', train=True, transform=transform, download=True)

# Izveido DataLoader treniņkopai
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Treniņkopas paraugu skaits:", len(train_dataset))  # Sagaidāms: 60000
print("Partiju (batches) skaits:", len(train_loader))

# Pārbauda vienu partiju
images, labels = next(iter(train_loader))
print("Partijas forma:", images.shape)   # Sagaidāms: (32, 1, 28, 28) -> (partija, kanāli, augstums, platums)
print("Etiķešu forma:", labels.shape)

## 8. Data Augmentation & Preprocessing

**Data augmentation** artificially increases the diversity of your training set by applying random transformations — without collecting new data. Common benefits:
- Reduces overfitting
- Makes the model more robust to real-world variation (different lighting, orientations, etc.)

⚠️ Important: augmentations should only be applied to the **training set**, not to validation/test sets.

Fill in the missing transform names:

In [ ]:
import torchvision.transforms as transforms

# Transformāciju konveijers treniņkopai ar datu paplašināšanu
train_transform = transforms.Compose([
    # Maina visu attēlu izmēru uz vienādu telpisko izmēru
    transforms.Resize((128, 128)),         
    # Ar 50% varbūtību apgriež attēlu horizontāli
    transforms.RandomHorizontalFlip(),                   
    # Nejauši rotē attēlu par ±15 grādiem
    transforms.RandomRotation(degrees=15),         
    # Nejauši maina spilgtumu un kontrastu
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  
    # Pārvērš PIL attēlu par float tenzoru [0, 1]
    transforms.ToTensor(),                   
    # Normalizē tenzoru uz [-1, 1]
    transforms.Normalize(mean=[0.5], std=[0.5])  
])

# Validācijas transformācija — bez nejaušām izmaiņām, tikai izmēra maiņa un normalizācija
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

### Applying Transforms and Exploring the Dataset

Let's make this concrete. Below we apply three different transform pipelines to MNIST and compare what comes out:

1. **No transform** — raw PIL image
2. **Minimal** — `ToTensor()` only
3. **Full augmentation** — resize, flip, rotation, normalise

We then inspect dataset size, batch shapes, pixel value ranges, and visualise a sample batch.

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ── 1. Bez transformācijas (neapstrādāts PIL attēls) ────────────────────────────────────────────────
raw_dataset = datasets.MNIST(root='data', train=True, transform=None, download=True)

# ── 2. Minimālā: tikai ToTensor ────────────────────────────────────────────────
minimal_transform = transforms.Compose([
    transforms.ToTensor()   # PIL [0,255] uint8 → float32 tenzors [0,1]
])
minimal_dataset = datasets.MNIST(root='data', train=True, transform=minimal_transform, download=False)

# ── 3. Pilns paplašināšanas konveijers ───────────────────────────────────────────
augment_transform = transforms.Compose([
    transforms.Resize((32, 32)),           # Palielina izmēru no 28×28 uz 32×32
    transforms.RandomHorizontalFlip(),     # Ar 50% varbūtību apgriež horizontāli
    transforms.RandomRotation(degrees=15), # Nejauši rotē par ±15 grādiem
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # [0,1] → [-1,1]
])
augmented_dataset = datasets.MNIST(root='data', train=True, transform=augment_transform, download=False)

# Ielādē arī testa datu kopu salīdzināšanai
test_dataset = datasets.MNIST(root='data', train=False, transform=minimal_transform, download=False)

# ── Datu kopu izmēri ──────────────────────────────────────
print("=== Datu kopu izmēri ===")
print(f"Treniņkopas paraugi : {len(minimal_dataset):,}") # MNIST treniņkopā ir 60,000 attēlu
print(f"Testa kopas paraugi  : {len(test_dataset):,}")   # MNIST testa kopā ir 10,000 attēlu
print(f"Kopā               : {len(minimal_dataset) + len(test_dataset):,}")
print(f"Klases             : {minimal_dataset.classes}")
print(f"Klašu skaits       : {len(minimal_dataset.classes)}")

In [ ]:
# ── Pārbauda atsevišķus paraugus ───────────────────────────────────────────────
# Iegūst pirmo paraugu (indekss 0) no katras datu kopas
print("=== Atsevišķu paraugu pārbaude ===")

raw_img, label = raw_dataset[0]
print(f"Neapstrādāts PIL attēls : tips={type(raw_img).__name__}, izmērs={raw_img.size}, etiķete={label}")

min_img, label = minimal_dataset[0]
print(f"Minimālais tenzors      : forma={min_img.shape}, tips={min_img.dtype}")
print(f"                        min={min_img.min():.3f}, max={min_img.max():.3f}")

aug_img, label = augmented_dataset[0]
print(f"Paplašinātais tenzors   : forma={aug_img.shape}, tips={aug_img.dtype}")
print(f"                        min={aug_img.min():.3f}, max={aug_img.max():.3f}")
print()
# J: Kāpēc paplašinātā tenzora minimālā vērtība ir negatīva, bet minimālā tenzora minimālā vērtība ir 0?
# A: Tāpēc, ka paplašināšanas konveijerā tiek izmantota normalizācija (transforms.Normalize), kas pārveido pikseļu vērtību diapazonu no [0, 1] uz [-1, 1]. Minimālajā konveijerā ir tikai ToTensor(), kas pārveido diapazonu uz [0, 1].

In [ ]:
# ── DataLoader partiju formas ──────────────────────────────────────────────────
# Izveido ielādētājus abiem konveijeriem
# shuffle=True ir svarīgi trenēšanas laikā, lai modelis katrā epohā redzētu datus citā secībā
loader_minimal   = DataLoader(minimal_dataset,   batch_size=32, shuffle=True)
loader_augmented = DataLoader(augmented_dataset, batch_size=32, shuffle=True)

imgs_min, labels = next(iter(loader_minimal))
print(f"Minimālā partija   — attēli: {imgs_min.shape}, etiķetes: {labels.shape}")
# Sagaidāmā forma: (32, 1, 28, 28) - (partijas izmērs, kanāli, augstums, platums)

imgs_aug, labels = next(iter(loader_augmented))
print(f"Paplašinātā partija — attēli: {imgs_aug.shape}, etiķetes: {labels.shape}")
# Sagaidāmā forma: (32, 1, 32, 32) - (partijas izmērs, kanāli, augstums, platums) - augstums/platums ir 32 dēļ Resize

print(f"\nPartiju skaits epohā (partijas izmērs=32): {len(loader_minimal)}")

# Klašu sadalījums vienā partijā
unique, counts = torch.unique(labels, return_counts=True)
print("\nKlašu sadalījums vienā partijā:")
for cls, cnt in zip(unique.tolist(), counts.tolist()):
    print(f"  Klase {cls}: {cnt} paraugs(i)")

In [ ]:
# ── Vizualizē partiju ────────────────────────────────────────────────────────
imgs, labels = next(iter(loader_augmented))

# Izveido 2x8 režģi attēlu attēlošanai
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle("Partijas paraugs — paplašināts MNIST", fontsize=11)

for i, ax in enumerate(axes.flat):
    # Paņem vienu attēlu un noņem kanāla dimensiju (1)
    img = imgs[i].squeeze().numpy()
    
    # Attēli ir normalizēti diapazonā [-1, 1]. Atceļ normalizāciju, lai tos varētu attēlot.
    # Formula: sākotnējais = pikselis * std + mean = pikselis * 0.5 + 0.5
    img = (img * 0.5) + 0.5    
    
    # Attēlo attēlu pelēktoņos
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    # Iestata virsrakstu ar attēla etiķeti
    ax.set_title(str(labels[i].item()), fontsize=9)
    # Noņem asis
    ax.axis('off')

plt.tight_layout()
plt.show()

## Summary

| Concept | Key class / function |
|---------|---------------------|
| Create tensor | `torch.tensor()`, `torch.rand()`, `torch.zeros()`, `torch.ones()` |
| Reshape tensor | `.reshape()`, `.view()`, `.squeeze()`, `.unsqueeze()`, `.permute()` |
| Move to GPU | `.to(device)` |
| Custom dataset | Subclass `Dataset`, implement `__len__` and `__getitem__` |
| Load in batches | `DataLoader(dataset, batch_size=…, shuffle=…)` |
| Preprocessing | `transforms.Compose([…])` |

**Next notebook:** building neural networks with `nn.Module` and writing a training loop.